# Nova-ST

## Import packages

In [ ]:
import pandas as pd
import numpy as np
import dask as da
import os
import gc
import matplotlib
from matplotlib import pyplot as plt
import json
import glob

In [ ]:
dataset_id = "Exp65-sampleSPC014"

path_novascope = f"/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/{dataset_id}/NovaScope/novascope_output"
seq1st = "NovaSeq6000_20241107"
chip_id = "03D1"

#### fastq2sbcd

In [ ]:
sbcds_manifest = pd.read_csv(os.path.join(path_novascope, f'seq1st/{seq1st}/sbcds/{seq1st}/manifest.tsv'), delimiter='\t')
sbcds_manifest
# NOTE: sbcds contains the information for ALL chips that were cut from this flowcell (not just the one used for this sample).

,id,filepath,barcodes,matches,mismatches,xmin,xmax,ymin,ymax
0,1_1101,1_1101.sbcds.sorted.tsv.gz,3441661,3377626,64035,1136,32931,1016,37059
1,1_1102,1_1102.sbcds.sorted.tsv.gz,3439440,3377026,62414,1136,32931,1016,37059
2,1_1103,1_1103.sbcds.sorted.tsv.gz,3440741,3377136,63605,1127,32922,1016,37059
3,1_1104,1_1104.sbcds.sorted.tsv.gz,3441762,3378802,62960,1136,32931,1016,37059
4,1_1105,1_1105.sbcds.sorted.tsv.gz,3441644,3375816,65828,1118,32922,1016,37059
...,...,...,...,...,...,...,...,...,...
3738,4_2674,4_2674.sbcds.sorted.tsv.gz,3423140,3357938,65202,1072,32859,1031,37059
3739,4_2675,4_2675.sbcds.sorted.tsv.gz,3414367,3347807,66560,1063,32859,1031,37059
3740,4_2676,4_2676.sbcds.sorted.tsv.gz,3414774,3355234,59540,1054,32850,1047,37059
3741,4_2677,4_2677.sbcds.sorted.tsv.gz,3419366,3355813,63553,1054,32841,1047,37059


In [ ]:
print('Number of tiles:', sbcds_manifest['id'].nunique())
print(f"Sum of all spatial barcodes: {sbcds_manifest['barcodes'].sum():,}")
print(f"Sum of all spatial barcodes (matches): {sbcds_manifest['matches'].sum():,}")
print(f"Sum of all spatial barcodes (mismatches): {sbcds_manifest['mismatches'].sum():,}") 

Number of tiles: 3743
Sum of all spatial barcodes: 12,827,831,696
Sum of all spatial barcodes (matches): 12,588,709,400
Sum of all spatial barcodes (mismatches): 239,122,296


#### sbcds2chip

In [ ]:
nbcds_manifest = pd.read_csv(os.path.join(path_novascope, f'seq1st/{seq1st}/nbcds/{chip_id}/manifest.tsv'), delimiter='\t')
nbcds_manifest
# NOTE: nbcds contains only the information for the chip of this particular sample.

,id,filepath,barcodes,matches,mismatches,xmin,xmax,ymin,ymax
0,1_1,1_1.sbcds.sorted.tsv.gz,117071907,117071907,0,0,11545568,0,6921220


In [ ]:
print(f"All spatial barcodes: {nbcds_manifest['barcodes'][0]:,}")
print(f"All spatial barcodes (matches): {nbcds_manifest['matches'][0]:,}")
print(f"All spatial barcodes (mismatches): {nbcds_manifest['mismatches'][0]:,}")
print(f"Coordinates (xmin, xmax, ymin, ymax): ({nbcds_manifest['xmin'][0]}, {nbcds_manifest['xmax'][0]}, {nbcds_manifest['ymin'][0]}, {nbcds_manifest['ymax'][0]})")

All spatial barcodes: 117,071,907
All spatial barcodes (matches): 117,071,907
All spatial barcodes (mismatches): 0
Coordinates (xmin, xmax, ymin, ymax): (0, 11545568, 0, 6921220)


In [ ]:
nbcds_sorted = pd.read_csv(os.path.join(path_novascope, f'seq1st/{seq1st}/nbcds/{chip_id}/1_1.sbcds.sorted.tsv.gz'), delimiter='\t', header=None, names=['barcode', 'lane_id', 'tile_id', 'x', 'y', 'count'])
nbcds_sorted
# This file maps spatial barcodes to global chip coordinates, with duplicate barcodes removed.

,barcode,lane_id,tile_id,x,y,count
0,AAAAAAAAAAAAAAAAACTGGGAAGGGTTATA,1,1,93662,2391507,1
1,AAAAAAAAAAAAAAAACGCGACTCTGGAAATT,1,1,2686233,4096979,1
2,AAAAAAAAAAAACACGACCGCATCTGGCGTTC,1,1,1061381,494606,1
3,AAAAAAAAAACGCGACCGGGGGGGGGCTTTAG,1,1,10951074,2969446,1
4,AAAAAAAAAACGGCGGGGTAGGAAATATGTGA,1,1,1578212,2597787,1
...,...,...,...,...,...,...
117071902,TTTTTTTTTTTTTTTTTTTTTTTTACTCCCCC,1,1,1707280,5342451,9
117071903,TTTTTTTTTTTTTTTTTTTTTTTTGGGGGCCA,1,1,9687621,2502838,8
117071904,TTTTTTTTTTTTTTTTTTTTTTTTTATTTTTT,1,1,10517958,2029621,9
117071905,TTTTTTTTTTTTTTTTTTTTTTTTTCAAAACA,1,1,10344267,6492626,8


In [ ]:
print(f"Number of unique spatial barcodes in capture area: {nbcds_sorted['barcode'].nunique():,}")
print(f"Length of spatial barcode: {len(nbcds_sorted['barcode'][0])} bp")
print(f"Number of lanes: {nbcds_sorted['lane_id'].nunique()}") # This probably does not accurately reflect the number of lanes
print(f"Number of tiles: {nbcds_sorted['tile_id'].nunique()}") # This clearly does not accurately reflect the number of tiles
print(f"Coordinates (xmin, xmax, ymin, ymax): ({nbcds_sorted['x'].min()}, {nbcds_sorted['x'].max()}, {nbcds_sorted['y'].min()}, {nbcds_sorted['y'].max()})")
print(f"Number of unique XY coordinates: {nbcds_sorted[['x', 'y']].drop_duplicates().shape[0]:,}")

Number of unique spatial barcodes in capture area: 117,071,907
Length of spatial barcode: 32 bp
Number of lanes: 1
Number of tiles: 1
Coordinates (xmin, xmax, ymin, ymax): (0, 11545568, 0, 6921220)
Number of unique XY coordinates: 117,071,907


#### smatch

In [ ]:
sum_reads = 0
sum_mismatches = 0
sum_matches = 0

for summary in glob.glob(os.path.join(path_novascope, f"match/{seq1st}/{chip_id}/**/*.summary.tsv"), recursive=True):
    df = pd.read_csv(summary, delimiter='\t', index_col='Type')
    sum_reads += df.at['Total', 'Reads']
    sum_mismatches += df.at['Miss', 'Reads']
    sum_matches += df.at['Match', 'Reads']

print(f"Sum of all reads: {sum_reads:,}")
print(f"Sum of all mismatches: {sum_mismatches:,} ({(sum_mismatches/sum_reads)*100:.1f} %)")
print(f"Sum of all matches: {sum_matches:,} ({(sum_matches/sum_reads)*100:.1f} %)") # These reads are used for the `align` step

# Total: All reads in the 2nd-seq FASTQ file.
# Miss: Reads that do not contain matching spatial barcodes.
# Match: Reads that match with a spatial barcode.
# Unique: Unique spatial barcodes that has matches.
# Dup(Exact): Duplicate barcodes calculated as Match - Unique.

Sum of all reads: 320,016,940
Sum of all mismatches: 178,588,952 (55.8 %)
Sum of all matches: 141,427,988 (44.2 %)


In [ ]:
unique = set()
unique_xy = set()
sum_count = 0
sum_mismatches = 0

for path in glob.glob(os.path.join(path_novascope, f"match/{seq1st}/{chip_id}/**/*.match.sorted.uniq.tsv.gz")):
    for chunk in pd.read_csv(path, sep="\t", header=None, names=['barcode', 'lane_id', 'tile_id', 'x', 'y', 'mismatches', 'count'], usecols=[0, 3, 4, 5, 6], chunksize=500000):
        unique.update(chunk['barcode'])
        unique_xy.update(zip(chunk['x'], chunk['y']))
        sum_count += chunk['count'].sum()
        sum_mismatches += chunk['mismatches'].sum()

print(f"Number of unique spatial barcodes: {len(unique):,}")
print(f"Number of unique XY coordinates: {len(unique_xy):,}")
print(f"Sum of spatial barcode counts (i.e. number of reads containing a valid spatial barcode): {sum_count:,}") # These reads are used for the `align` step
print(f"Sum of mismatches across unique spatial barcodes: {sum_mismatches:,}")

Number of unique spatial barcodes: 30,952,152
Number of unique XY coordinates: 30,952,152
Sum of spatial barcode counts (i.e. number of reads containing a valid spatial barcode): 141,427,988
Sum of mismatches across unique spatial barcodes: 220,631


#### align

In [ ]:
# STARsolo barcode statistics
bc_stats = {}
with open(os.path.join(path_novascope, f"align/{seq1st}/{chip_id}/{dataset_id}/bam/sttoolsSolo.out/Barcodes.stats")) as f:
    for line in f:
        line = line.strip().split()
        bc_stats[line[0]] = int(line[1])
        
total = sum(bc_stats.values())

print(f"noNoAdapter: {bc_stats['noNoAdapter']:,} ({bc_stats['noNoAdapter'] / total * 100:.1f}%)",
    "  -   Number of reads without an adapter")
print(f"noNoUMI: {bc_stats['noNoUMI']:,} ({bc_stats['noNoUMI'] / total * 100:.1f}%)",
    "  -   Number of reads not associated with a valid UMI")
print(f"noNoCB: {bc_stats['noNoCB']:,} ({bc_stats['noNoCB'] / total * 100:.1f}%)",
    "  -   Number of reads not associated with a valid spatial barcode")
print(f"noNinCB: {bc_stats['noNinCB']:,} ({bc_stats['noNinCB'] / total * 100:.1f}%)",
    "  -   Number of reads with too many Ns in the spatial barcode")
print(f"noNinUMI: {bc_stats['noNinUMI']:,} ({bc_stats['noNinUMI'] / total * 100:.1f}%)",
    "  -   Number of reads with too many Ns in the UMI")
print(f"noUMIhomopolymer: {bc_stats['noUMIhomopolymer']:,} ({bc_stats['noUMIhomopolymer'] / total * 100:.1f}%)",
    "  -   Number of reads that have a homopolymeric UMI")
print(f"noNoWLmatch: {bc_stats['noNoWLmatch']:,} ({bc_stats['noNoWLmatch'] / total * 100:.1f}%)",
    "  -   Number of reads that do not have a match to the whitelist (even when allowing one mismatch)")
print(f"noTooManyMM: {bc_stats['noTooManyMM']:,} ({bc_stats['noTooManyMM'] / total * 100:.1f}%)",
    "  -   Number of reads that have too many mismatches to the whitelist")
print(f"noTooManyWLmatches: {bc_stats['noTooManyWLmatches']:,} ({bc_stats['noTooManyWLmatches'] / total * 100:.1f}%)",
    "  -   Number of reads that have too many matches to the whitelist (not used at the moment)")
print(f"yesWLmatchExact: {bc_stats['yesWLmatchExact']:,} ({bc_stats['yesWLmatchExact'] / total * 100:.1f}%)",
    "  -   Number of reads with an exact match to the whitelist")
print(f"yesOneWLmatchWithMM: {bc_stats['yesOneWLmatchWithMM']:,} ({bc_stats['yesOneWLmatchWithMM'] / total * 100:.1f}%)",
    "  -   Number of reads with one match to the whitelist but there are mismatches")
print(f"yesMultWLmatchWithMM: {bc_stats['yesMultWLmatchWithMM']:,} ({bc_stats['yesMultWLmatchWithMM'] / total * 100:.1f}%)",
    "  -   Number of reads with multiple matches to the whitelist but there are mismatches")

print(f"\nSum of all bc_stats: {total:,} (100.00%)")

yes_sum = (
    bc_stats['yesWLmatchExact']
    + bc_stats['yesOneWLmatchWithMM']
    + bc_stats['yesMultWLmatchWithMM']
)

print(f"Sum of `yes` bc_stats: {yes_sum:,} ({yes_sum / total * 100:.2f}%)")

# For explanation about stats see: https://github.com/alexdobin/STAR/issues/1887
# prefix 'no' means that reads are not used for quantification

noNoAdapter: 0 (0.0%)   -   Number of reads without an adapter
noNoUMI: 0 (0.0%)   -   Number of reads not associated with a valid UMI
noNoCB: 0 (0.0%)   -   Number of reads not associated with a valid spatial barcode
noNinCB: 2 (0.0%)   -   Number of reads with too many Ns in the spatial barcode
noNinUMI: 69,116 (0.0%)   -   Number of reads with too many Ns in the UMI
noUMIhomopolymer: 495,737 (0.4%)   -   Number of reads that have a homopolymeric UMI
noNoWLmatch: 188,338 (0.1%)   -   Number of reads that do not have a match to the whitelist (even when allowing one mismatch)
noTooManyMM: 0 (0.0%)   -   Number of reads that have too many mismatches to the whitelist
noTooManyWLmatches: 0 (0.0%)   -   Number of reads that have too many matches to the whitelist (not used at the moment)
yesWLmatchExact: 138,394,666 (97.9%)   -   Number of reads with an exact match to the whitelist
yesOneWLmatchWithMM: 94 (0.0%)   -   Number of reads with one match to the whitelist but there are mismatches


In [ ]:
# STARsolo feature statistics (Gene)
feature_stats = {}
with open(os.path.join(path_novascope, f"align/{seq1st}/{chip_id}/{dataset_id}/bam/sttoolsSolo.out/Gene/Features.stats")) as f:
    for line in f:
        line = line.strip().split()
        feature_stats[line[0]] = int(line[1])
    
print(f"noUnmapped: {feature_stats['noUnmapped']:,} ({feature_stats['noUnmapped'] / yes_sum * 100:.1f}%)  -  Number of reads that do not map to the reference genome")
print(f"noNoFeature: {feature_stats['noNoFeature']:,} ({feature_stats['noNoFeature'] / yes_sum * 100:.1f}%)  -  Number of reads that map to the genome but not to any feature")
print(f"MultiFeature: {feature_stats['MultiFeature']:,} ({feature_stats['MultiFeature'] / yes_sum * 100:.1f}%)  -  Number of reads that map to multiple features")
print(f"    - subMultiFeatureMultiGenomic: {feature_stats['subMultiFeatureMultiGenomic']:,} ({feature_stats['subMultiFeatureMultiGenomic'] / yes_sum * 100:.1f}%)  -  Reads mapping to multiple genomic loci and multiple features (a subset of MultiFeature)")
print(f"noTooManyWLmatches: {feature_stats['noTooManyWLmatches']:,} ({feature_stats['noTooManyWLmatches'] / yes_sum * 100:.1f}%)  -  Number of reads rejected because the spatial barcode has too many matches to the whitelist")
print(f"noMMtoWLwithoutExact: {feature_stats['noMMtoWLwithoutExact']:,} ({feature_stats['noMMtoWLwithoutExact'] / yes_sum * 100:.1f}%)  -  Number of reads rejected due to mismatches without an exact-match backup")
print(f"yesWLmatch: {feature_stats['yesWLmatch']:,} ({feature_stats['yesWLmatch'] / yes_sum * 100:.1f}%)  -  Reads mapping to a feature with a whitelist-matching barcode")
print(f"    - yessubWLmatchExact: {feature_stats['yessubWLmatchExact']:,} ({feature_stats['yessubWLmatchExact'] / yes_sum * 100:.1f}%)  -  Exact whitelist matches (subset of yesWLmatch)")
print(f"    - yessubWLmatch_UniqueFeature: {feature_stats['yessubWLmatch_UniqueFeature']:,} ({feature_stats['yessubWLmatch_UniqueFeature'] / yes_sum * 100:.1f}%)  -  Reads mapping uniquely to a feature (subset of yesWLmatch)")

print(f"\nyesCellBarcodes: {feature_stats['yesCellBarcodes']:,}  -  Number of unique valid spatial barcodes (i.e. Number of spots)")
print(f"yesUMIs: {feature_stats['yesUMIs']:,}  -    Number of unique valid UMIs")

mapped_sum = (
    feature_stats['yesWLmatch']
    + feature_stats['noNoFeature']
    + feature_stats['noUnmapped']
    + feature_stats['MultiFeature']
    + feature_stats['noTooManyWLmatches']
    + feature_stats['noMMtoWLwithoutExact']
)

print(f"\nSum of all mapped, unmapped and multi-mapped reads: {mapped_sum:,} ({mapped_sum / yes_sum * 100:.1f}%)")

# For explanation about stats see: https://github.com/alexdobin/STAR/issues/1887, https://github.com/alexdobin/STAR/blob/master/doc/STARmanual.pdf and https://training.galaxyproject.org/training-material/topics/single-cell/tutorials/scrna-preprocessing-tenx/tutorial.html. 
# The explanations above are my best guesses based on the available information.
# NOTE: In addition to the gene counts (Gene; exons), STARsolo also calculates counts for the following features:
#   - pre-mRNA counts (GeneFull) (exons + introns)
#   - annotated and novel splice junctions (SJ)
#   - Velocyto spliced/unspliced/ambiguous counts (Velocyto)

noUnmapped: 16,353,708 (11.6%)  -  Number of reads that do not map to the reference genome
noNoFeature: 22,021,193 (15.7%)  -  Number of reads that map to the genome but not to any feature
MultiFeature: 1,579,760 (1.1%)  -  Number of reads that map to multiple features
    - subMultiFeatureMultiGenomic: 1,292,468 (0.9%)  -  Reads mapping to multiple genomic loci and multiple features (a subset of MultiFeature)
noTooManyWLmatches: 1,332,546 (0.9%)  -  Number of reads rejected because the spatial barcode has too many matches to the whitelist
noMMtoWLwithoutExact: 1 (0.0%)  -  Number of reads rejected due to mismatches without an exact-match backup
yesWLmatch: 99,387,587 (70.7%)  -  Reads mapping to a feature with a whitelist-matching barcode
    - yessubWLmatchExact: 99,106,668 (70.5%)  -  Exact whitelist matches (subset of yesWLmatch)
    - yessubWLmatch_UniqueFeature: 99,387,587 (70.7%)  -  Reads mapping uniquely to a feature (subset of yesWLmatch)

yesCellBarcodes: 22,209,387  -  Numb

In [ ]:
# Summary statistics (Gene)
summary = pd.read_csv(os.path.join(path_novascope, f"align/{seq1st}/{chip_id}/{dataset_id}/bam/sttoolsSolo.out/Gene/Summary.csv"), sep=",", index_col=0, header=None)
print(f"Number of reads used for STARsolo: {int(summary.loc['Number of Reads'].values[0]):,}")
print(f"Sequencing saturation: {float(summary.loc['Sequencing Saturation'].values[0])*100:.1f} %")
print(f"Q30 bases in spatial barcode + UMI: {float(summary.loc['Q30 Bases in CB+UMI'].values[0])*100:.1f} %")
print(f"Q30 bases in RNA read: {float(summary.loc['Q30 Bases in RNA read'].values[0])*100:.1f} %")

Number of reads used for STARsolo: 141,427,988
Sequencing saturation: 54.9 %
Q30 bases in spatial barcode + UMI: 95.1 %
Q30 bases in RNA read: 91.9 %


#### dge2sdge

In [ ]:
path_transcripts = os.path.join(path_novascope,f"analysis/{dataset_id}/{dataset_id}-default/preprocess/{dataset_id}-default.transcripts.tsv.gz")

transcripts = pd.read_csv(path_transcripts, delimiter='\t') # Does not contain spatial barcodes
transcripts

,#lane,tile,X,Y,gene_id,gene,gn,gt,spl,unspl,ambig
0,0,0,0,0,ENSMUSG00000000088,Cox5a,1,1,1,0,0
1,0,0,0,0,ENSMUSG00000000088,Cox5a,1,1,1,0,0
2,0,0,0,0,ENSMUSG00000000171,Sdhd,1,1,1,0,0
3,0,0,0,0,ENSMUSG00000000247,Lhx2,1,1,1,0,0
4,0,0,0,0,ENSMUSG00000000346,Dazap2,1,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...
46192401,1,1,11544455,4420538,ENSMUSG00000032239,Rp9,1,1,0,0,1
46192402,1,1,11544455,4420538,ENSMUSG00000064354,mt-Co2,1,1,0,0,0
46192403,1,1,11545012,4542859,ENSMUSG00000038976,Ppp1r9b,1,1,1,0,0
46192404,1,1,11545568,3874700,ENSMUSG00000022982,Sod1,1,1,1,0,0


In [ ]:
print(f"Coordinates (xmin, xmax, ymin, ymax): ({transcripts['X'].min()}, {transcripts['X'].max()}, {transcripts['Y'].min()}, {transcripts['Y'].max()})")
print(f"Number of unique XY coordinates: {transcripts[['X', 'Y']].drop_duplicates().shape[0]:,}")
print(f"Sum of gn: {transcripts['gn'].sum():,}")
print(f"Sum of gt: {transcripts['gt'].sum():,}")
print(f"Sum of spl: {transcripts['spl'].sum():,}")
print(f"Sum of unspl: {transcripts['unspl'].sum():,}")
print(f"Sum of ambig: {transcripts['ambig'].sum():,}")

Coordinates (xmin, xmax, ymin, ymax): (0, 11545568, 0, 6921220)
Number of unique XY coordinates: 23,466,428
Sum of gn: 44,793,966
Sum of gt: 46,858,743
Sum of spl: 34,167,009
Sum of unspl: 1,329,662
Sum of ambig: 2,109,630
